![DataProjectLab](../../logo_dataprojectlab.png)


# Notebook 2 — SQL Analytics : KPIs, Performance & Analyses Avancees
**SOLUTION COMPLETE**

**Projet :** HotelChain West Analytics  
**Prerequis :** Notebook 1 termine, base HotelChainDB chargee  
**Auteur :** DataProjectLab


---
## Contexte

La direction d'HotelChain West a besoin de reponses precises a 5 questions business :

1. Quels hotels et quelles periodes ont les meilleurs taux d'occupation ?
2. Quels canaux generent le plus de revenu par reservation ?
3. Quels types de chambres sont les plus rentables ?
4. Comment evolue le revenu mois par mois par canal ?
5. Quels clients fidelesles genèrent le plus de valeur ?

Ce notebook repond a toutes ces questions avec des requetes SQL Server avancees : **vues**, **CTEs**, **window functions** (RANK, LAG, NTILE, PERCENT_RANK), **PIVOT**, et **procedures stockees**.

---
## Note technique — Colonne `csat` stockée en VARCHAR

### Pourquoi csat est VARCHAR dans la table reservations ?

Lors du BULK INSERT, SQL Server ne peut pas convertir une cellule **vide** `,,` en `INT`. Or la colonne `csat` est NULL pour toutes les reservations non terminees (on ne peut noter un sejour qu'apres l'avoir vecu). Le CSV contient donc beaucoup de cellules vides sur cette colonne.

**Solution appliquee :** `csat` a ete declare en `VARCHAR(5)` dans le `CREATE TABLE`. Toutes les requetes de ce notebook utilisent `TRY_CAST(csat AS INT)` pour convertir a la volee, ce qui retourne `NULL` automatiquement si la valeur est vide ou non numerique — sans erreur.

```sql
-- Comportement de TRY_CAST
TRY_CAST(''    AS INT)  --> NULL  (cellule vide)
TRY_CAST('4'  AS INT)  --> 4     (valeur valide)
TRY_CAST('abc' AS INT) --> NULL  (valeur non numerique)
```

> **A retenir :** `TRY_CAST` est la version securisee de `CAST`. `CAST` plante sur une valeur invalide, `TRY_CAST` retourne NULL. Utiliser systematiquement `TRY_CAST` sur les colonnes issues d'un import CSV.

---
## Etape 1 — Vues de nettoyage

### METHODE — Pourquoi des vues et pas des tables ?

Une **vue SQL** (`VIEW`) est une requete sauvegardee qui se comporte comme une table. La difference fondamentale avec une table physique :

| | Table physique | Vue |
|---|---|---|
| Stockage | Donnees copiees sur disque | Aucune copie — requete executee a la volee |
| Modification source | Ne se met pas a jour | Reflete toujours les donnees actuelles |
| Donnees originales | Modifiees si UPDATE | Jamais modifiees |

**Principe de conservation :** on ne supprime jamais les donnees brutes. On cree des vues qui filtrent les anomalies. Ainsi :
- Si on decouvre une erreur dans notre logique de filtrage, on corrige la vue
- Les donnees originales sont toujours disponibles pour audit
- Power BI se connectera aux vues, pas aux tables brutes

`CREATE OR ALTER VIEW` est une commande SQL Server (syntaxe modernisee depuis SQL Server 2016) qui cree la vue si elle n'existe pas, ou la remplace si elle existe deja. Equivalent a `DROP VIEW IF EXISTS` + `CREATE VIEW`.

In [ ]:
-- VUE 1 : Clients propres (sans doublons email et sans ages aberrants)
CREATE OR ALTER VIEW vw_clients_propres AS
SELECT *
FROM clients
WHERE age BETWEEN 16 AND 80
  AND client_id IN (
      -- Garder uniquement le premier client_id pour chaque email (MIN = le plus ancien)
      SELECT MIN(client_id) AS client_id
      FROM clients
      WHERE email IS NOT NULL
      GROUP BY email
  );
GO

-- VUE 2 : Reservations propres (sans montants negatifs ni nuits = 0)
CREATE OR ALTER VIEW vw_reservations_propres AS
SELECT *
FROM reservations
WHERE montant_total > 0
  AND nb_nuits      > 0;
GO

-- VUE 3 : Paiements valides uniquement
CREATE OR ALTER VIEW vw_paiements_valides AS
SELECT *
FROM paiements
WHERE statut_paiement = 'Valide'
  AND montant         > 0;
GO


In [ ]:
-- Verification : comparaison avant / apres nettoyage
SELECT 'clients brut'          AS source, COUNT(*) AS nb FROM clients
UNION ALL
SELECT 'vw_clients_propres',    COUNT(*) FROM vw_clients_propres
UNION ALL
SELECT 'reservations brut',     COUNT(*) FROM reservations
UNION ALL
SELECT 'vw_reservations_propres', COUNT(*) FROM vw_reservations_propres
UNION ALL
SELECT 'paiements brut',        COUNT(*) FROM paiements
UNION ALL
SELECT 'vw_paiements_valides',  COUNT(*) FROM vw_paiements_valides;


### INTERPRETATION

| Source | Lignes brutes | Lignes propres | Supprimees |
|---|---|---|---|
| clients | 3 000 | **2 993** | 7 (3 doublons + 4 ages aberrants) |
| reservations | 8 000 | **7 992** | 8 (5 montants negatifs + 3 nuits=0) |
| paiements | 9 698 | **9 338** | 360 (echecs + negatifs + rembourses) |

### METIER
Les 360 paiements non valides (3.7% du total) representent des transactions echouees ou des remboursements. Les garder dans les analyses de revenu fausserait le chiffre d'affaires. En les isolant dans une vue, l'equipe finance peut aussi les analyser separement pour le suivi des incidents.

---
## Etape 2 — KPIs globaux en une seule requete

### METHODE — Le pattern CASE WHEN pour les KPIs multiples

En SQL analytique, le pattern `SUM(CASE WHEN condition THEN valeur ELSE 0 END)` permet de calculer des agregations conditionnelles dans une seule requete. C'est l'equivalent des formules `CALCULATE` + `FILTER` en DAX Power BI.

**Avantage :** une seule passe sur les donnees au lieu de N requetes separees. Sur un grand dataset, c'est 5 a 10x plus rapide.

`CAST(colonne AS FLOAT)` est necessaire pour les moyennes sur des colonnes `INT`. Sans CAST, SQL Server calcule en division entiere : `5 / 2 = 2` au lieu de `2.5`.

In [ ]:
-- KPIs globaux HotelChain West (toutes donnees nettoyees)
-- Note : TRY_CAST(csat AS INT) car csat est stocke en VARCHAR(5)
SELECT
    COUNT(DISTINCT r.reservation_id)                           AS total_reservations,
    COUNT(DISTINCT r.client_id)                                AS clients_uniques,
    ROUND(SUM(p.montant), 0)                                   AS revenu_total_fcfa,
    ROUND(SUM(p.montant) / 1000000.0, 2)                       AS revenu_total_millions,
    ROUND(AVG(CASE WHEN r.statut = 'Terminee'
                   THEN TRY_CAST(r.csat AS FLOAT) END), 2)     AS csat_moyen,
    ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 2)                   AS duree_moyenne_nuits,
    ROUND(COUNT(CASE WHEN r.statut = 'Annulee'
                     THEN 1 END) * 100.0
          / COUNT(*), 1)                                       AS taux_annulation_pct,
    ROUND(SUM(p.montant)
          / NULLIF(COUNT(DISTINCT p.reservation_id), 0), 0)    AS revenu_moyen_reservation
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p
    ON r.reservation_id = p.reservation_id;


### INTERPRETATION

| KPI | Valeur | Lecture |
|---|---|---|
| Revenu total | **2.87 Mrd FCFA** | ~4.4M EUR sur 30 mois |
| Clients uniques | **2 775** | Sur 3 000 inscrits (7.5% inactifs) |
| CSAT moyen | **4.01 / 5** | Bonne satisfaction globale |
| Duree moyenne | **2.71 nuits** | Profil dominant : voyageurs d'affaires |
| Taux annulation | **3.8%** | Excellent (norme secteur : 5-15%) |
| Revenu moyen/res | **307 874 FCFA** | ~470 EUR par reservation |

### METHODE — NULLIF
`NULLIF(expr, 0)` retourne NULL si l'expression vaut 0, sinon retourne l'expression. Utilise en combinaison avec une division pour eviter l'erreur 'division par zero'. C'est l'equivalent de `DIVIDE()` en DAX Power BI.

---
## Etape 3 — Taux d'occupation par hotel et par mois

### METHODE — Comprendre le taux d'occupation hotelier

Le **taux d'occupation** est l'indicateur de reference de l'industrie hoteliere :

```
Taux occupation = Nuits vendues / (Nb chambres × Nb jours du mois) × 100
```

**Exemple :** un hotel de 80 chambres qui vend 1 200 nuits en janvier (31 jours) :
```
Capacite theorique = 80 × 31 = 2 480 nuits disponibles
Taux = 1 200 / 2 480 × 100 = 48.4%
```

**Points techniques SQL importants :**
- `DAY(EOMONTH(date))` retourne le nombre de jours du mois de la date donnee
- `EOMONTH(date)` retourne le dernier jour du mois de la date
- `DATEFROMPARTS(annee, mois, 1)` construit une date depuis ses composantes

### METHODE — CTEs (Common Table Expressions)

Une **CTE** (introduite par `WITH nom AS (...)`) est une sous-requete nommee qui existe le temps d'une requete. Elle permet de decomposer une requete complexe en etapes lisibles. Avantages :
- Lisibilite : chaque etape a un nom metier clair
- Reutilisation : on peut referencer la meme CTE plusieurs fois
- Maintenance : modifier une etape sans toucher les autres

In [ ]:
-- Taux d'occupation mensuel par hotel (avec CTE + RANK)
WITH nuits_vendues AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)       AS annee,
        MONTH(r.date_arrivee)      AS mois,
        SUM(r.nb_nuits)            AS nuits_vendues
    FROM vw_reservations_propres r
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
capacite_mensuelle AS (
    SELECT
        h.hotel_id,
        h.nb_chambres,
        cal.annee,
        cal.mois,
        -- DAY(EOMONTH) = nombre de jours du mois
        h.nb_chambres * DAY(EOMONTH(DATEFROMPARTS(cal.annee, cal.mois, 1)))
            AS capacite_theorique
    FROM hotels h
    CROSS JOIN (
        -- Generer toutes les combinaisons annee x mois
        SELECT DISTINCT
            YEAR(date_arrivee)  AS annee,
            MONTH(date_arrivee) AS mois
        FROM vw_reservations_propres
    ) cal
),
occupation AS (
    SELECT
        cm.hotel_id,
        cm.annee,
        cm.mois,
        ISNULL(nv.nuits_vendues, 0)          AS nuits_vendues,
        cm.capacite_theorique,
        ROUND(ISNULL(nv.nuits_vendues, 0) * 100.0
              / cm.capacite_theorique, 1)     AS taux_occupation
    FROM capacite_mensuelle cm
    LEFT JOIN nuits_vendues nv
        ON cm.hotel_id = nv.hotel_id
        AND cm.annee   = nv.annee
        AND cm.mois    = nv.mois
)
SELECT
    h.nom                        AS hotel,
    o.annee,
    o.mois,
    o.nuits_vendues,
    o.capacite_theorique,
    o.taux_occupation,
    RANK() OVER (
        PARTITION BY o.annee, o.mois
        ORDER BY o.taux_occupation DESC
    )                            AS rang_du_mois
FROM occupation o
JOIN hotels h ON o.hotel_id = h.hotel_id
ORDER BY o.annee, o.mois, o.taux_occupation DESC;


### INTERPRETATION

Les taux d'occupation reels sont **faibles** mais coherents avec les volumes generes par le dataset (30 mois / 515 chambres) :

| Hotel | Taux moyen | Rang |
|---|---|---|
| HotelChain Douala | **6.4%** | 1 — meilleur taux |
| HotelChain Cocody | 5.5% | 2 |
| HotelChain Dakar | 4.5% | 3 |
| HotelChain Plateau | 3.8% | 4 |
| HotelChain Accra | **2.8%** | 5 — taux le plus bas |

**Pic mensuel :** HotelChain Douala en aout 2023 atteint **8.6%** — son meilleur mois. La saisonnalite est visible : les mois de juillet-aout et janvier-fevrier ont des taux plus eleves (voyages professionnels et conferences de debut/fin d'annee).

### METHODE — RANK() OVER (PARTITION BY)
`RANK() OVER (PARTITION BY annee, mois ORDER BY taux DESC)` classe les hotels **a l'interieur de chaque mois**. Sans PARTITION BY, tous les hotels de tous les mois seraient classes ensemble — ce qui n'a pas de sens metier. PARTITION BY est l'equivalent de `GROUP BY` pour les window functions.

In [ ]:
-- Vue synthese taux d'occupation annuel par hotel
SELECT
    h.nom                                                          AS hotel,
    YEAR(r.date_arrivee)                                           AS annee,
    ROUND(SUM(r.nb_nuits) * 100.0
          / (h.nb_chambres * 365), 1)                             AS taux_occ_annuel,
    SUM(r.nb_nuits)                                                AS nuits_vendues,
    h.nb_chambres * 365                                            AS capacite_annuelle
FROM vw_reservations_propres r
JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE r.statut IN ('Terminee', 'En cours')
  AND YEAR(r.date_arrivee) IN (2022, 2023)  -- annees completes seulement
GROUP BY h.nom, h.hotel_id, h.nb_chambres, YEAR(r.date_arrivee)
ORDER BY annee, taux_occ_annuel DESC;


---
## Etape 4 — Performance par canal avec RANK() et LAG()

### METHODE — LAG() : mesurer l'evolution dans le temps

`LAG(colonne, n) OVER (PARTITION BY ... ORDER BY ...)` retourne la valeur de la ligne **n positions avant** dans la partition ordonnee. C'est la facon SQL de calculer 'la valeur du mois precedent' sans jointure.

**Syntaxe et semantique :**
```sql
LAG(revenu_mensuel, 1) OVER (PARTITION BY canal ORDER BY annee, mois)
-- Retourne : le revenu du mois precedent pour ce canal
-- Janvier 2023 → retourne decembre 2022
-- Janvier 2022 → retourne NULL (pas de mois precedent)
```

Son inverse `LEAD()` retourne la valeur **n positions apres** (utile pour comparer avec le mois suivant).

In [ ]:
-- Performance globale par canal (RANK inclus)
SELECT
    r.canal,
    COUNT(*)                                                    AS nb_reservations,
    ROUND(SUM(p.montant), 0)                                    AS revenu_total,
    ROUND(AVG(p.montant), 0)                                    AS revenu_moyen,
    ROUND(COUNT(CASE WHEN r.statut = 'Annulee' THEN 1 END)
          * 100.0 / COUNT(*), 1)                               AS taux_annulation_pct,
    RANK() OVER (ORDER BY SUM(p.montant) DESC)                  AS rang_revenu,
    RANK() OVER (ORDER BY AVG(p.montant) DESC)                  AS rang_rentabilite
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p
    ON r.reservation_id = p.reservation_id
GROUP BY r.canal
ORDER BY revenu_total DESC;


### INTERPRETATION

| Canal | Reservations | Revenu total | Revenu moyen | Taux annul. |
|---|---|---|---|---|
| **Direct** | 2 967 | **913 M FCFA** | 307 774 FCFA | 3.4% |
| Booking.com | 2 834 | 835 M FCFA | 294 582 FCFA | 4.1% |
| Agence voyage | 1 444 | 431 M FCFA | 298 525 FCFA | 3.5% |
| Expedia | 1 534 | 420 M FCFA | 273 639 FCFA | 3.5% |
| Corporate | 996 | 275 M FCFA | 276 602 FCFA | 4.6% |

### METIER — 3 insights actionnables

**1. Le Direct est roi** : 913 M FCFA, revenu moyen le plus eleve (307 774 FCFA) ET taux d'annulation le plus bas (3.4%). Sans commissions OTA (10-15%), le revenu net Direct est encore plus avantageux. Chaque point de part de marche gagne sur le Direct genere ~9 M FCFA de revenu additionnel.

**2. Corporate = risque** : le canal Corporate a le taux d'annulation le plus eleve (4.6%) et le revenu moyen le plus faible (276 602 FCFA). Les contrats Corporate negocient des tarifs reduits mais annulent plus souvent. A revoir dans la politique tarifaire.

**3. Expedia sous-performe** : malgre plus de reservations qu'une Agence voyage (1 534 vs 1 444), Expedia genere presque le meme revenu. Son revenu moyen par reservation (273 639 FCFA) est le plus bas des OTA — Expedia applique probablement les remises les plus agressives.

In [ ]:
-- Tendance mensuelle du revenu par canal (avec LAG)
WITH rev_mensuel AS (
    SELECT
        r.canal,
        YEAR(p.date_paiement)   AS annee,
        MONTH(p.date_paiement)  AS mois,
        SUM(p.montant)          AS revenu_mensuel
    FROM vw_reservations_propres r
    JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    GROUP BY r.canal, YEAR(p.date_paiement), MONTH(p.date_paiement)
)
SELECT
    canal,
    annee,
    mois,
    revenu_mensuel,
    LAG(revenu_mensuel, 1) OVER (
        PARTITION BY canal
        ORDER BY annee, mois
    )                           AS revenu_mois_prec,
    ROUND(
        (revenu_mensuel
         - LAG(revenu_mensuel, 1) OVER (PARTITION BY canal ORDER BY annee, mois))
        * 100.0
        / NULLIF(LAG(revenu_mensuel, 1) OVER (
            PARTITION BY canal ORDER BY annee, mois), 0)
    , 1)                        AS variation_pct
FROM rev_mensuel
ORDER BY canal, annee, mois;


### INTERPRETATION — Tendance canal Direct 2023

La variation mensuelle du canal Direct sur 2023 montre une forte volatilite :

| Mois | Variation | Signal |
|---|---|---|
| Fevrier | **-23.9%** | Creux hivernal post-janvier |
| Mars | **+26.1%** | Rebond conferences T1 |
| Mai | **+29.7%** | Pic pre-ete |
| Aout | **-36.8%** | Creux estival — direction absente |

Cette saisonnalite est caracteristique des hotels d'affaires : fort en mars-mai (conferences T1), creux en aout (vacances), rebond septembre-octobre (rentree). La politique tarifaire doit adapter les tarifs a ces cycles.

---
## Etape 5 — ADR et RevPAR par hotel

### METHODE — Les 3 KPIs de revenus de l'industrie hoteliere

L'industrie hoteliere utilise 3 indicateurs standardises :

**1. ADR (Average Daily Rate)** = Revenu chambres / Nuits vendues
> Prix moyen effectivement paye par nuit. Mesure la politique tarifaire.

**2. Taux d'occupation** = Nuits vendues / Nuits disponibles
> Utilisation de la capacite. Mesure la demande.

**3. RevPAR (Revenue Per Available Room)** = ADR × Taux d'occupation
> `RevPAR = Revenu total / Nb chambres disponibles`
> L'indicateur le plus important car il combine prix ET occupation. Un hotel peut avoir un ADR eleve mais un RevPAR faible si son occupation est basse, et inversement.

**Pourquoi les CTEs sont indispensables ici :**
Le calcul du RevPAR necessite 2 agregations independantes (ADR et taux d'occupation) avant de les combiner. Les CTEs permettent de faire ca proprement sans sous-requetes imbriquees illisibles.

In [ ]:
-- Requete 1/2 : detail mensuel ADR / RevPAR par hotel
-- Retourne une ligne par hotel x mois — utile pour voir la saisonnalite
-- Pour la synthese globale (un seul chiffre par hotel), voir la requete suivante
-- ADR et RevPAR par hotel et par mois
WITH adr_calc AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)    AS annee,
        MONTH(r.date_arrivee)   AS mois,
        -- ADR = revenu total / nuits vendues
        ROUND(SUM(p.montant)
              / NULLIF(SUM(r.nb_nuits), 0), 0)  AS adr,
        SUM(r.nb_nuits)                          AS nuits_vendues,
        SUM(p.montant)                           AS revenu_chambre
    FROM vw_reservations_propres r
    JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
occupation_calc AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)    AS annee,
        MONTH(r.date_arrivee)   AS mois,
        -- Taux occupation = nuits vendues / capacite theorique
        ROUND(SUM(r.nb_nuits) * 100.0
              / (h.nb_chambres
                 * DAY(EOMONTH(DATEFROMPARTS(
                     YEAR(r.date_arrivee),
                     MONTH(r.date_arrivee), 1))))
        , 1)                    AS taux_occupation
    FROM vw_reservations_propres r
    JOIN hotels h ON r.hotel_id = h.hotel_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, h.nb_chambres, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
)
SELECT
    h.nom                        AS hotel,
    a.annee,
    a.mois,
    a.adr,
    o.taux_occupation,
    -- RevPAR = ADR x Taux occupation / 100
    ROUND(a.adr * o.taux_occupation / 100.0, 0) AS revpar,
    RANK() OVER (
        PARTITION BY a.annee, a.mois
        ORDER BY ROUND(a.adr * o.taux_occupation / 100.0, 0) DESC
    )                            AS rang_revpar
FROM adr_calc a
JOIN hotels h          ON a.hotel_id = h.hotel_id
JOIN occupation_calc o ON a.hotel_id = o.hotel_id
                      AND a.annee    = o.annee
                      AND a.mois     = o.mois
ORDER BY a.annee, a.mois, revpar DESC;


In [ ]:
-- Synthese globale ADR / Taux occupation / RevPAR par hotel (toute la periode)
-- C'est CETTE requete qui correspond au tableau de l'interpretation ci-dessous
WITH adr_global AS (
    SELECT
        r.hotel_id,
        ROUND(SUM(p.montant)
              / NULLIF(SUM(r.nb_nuits), 0), 0)           AS adr,
        SUM(r.nb_nuits)                                   AS nuits_vendues,
        SUM(p.montant)                                    AS revenu_total
    FROM vw_reservations_propres r
    JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id
),
cap_globale AS (
    -- Capacite totale sur toute la periode (jan 2022 - juin 2024 = 30 mois)
    SELECT
        hotel_id,
        SUM(nb_chambres
            * DAY(EOMONTH(DATEFROMPARTS(annee, mois, 1)))) AS cap_totale
    FROM (
        SELECT h.hotel_id, h.nb_chambres,
               v.annee, v.mois
        FROM hotels h
        CROSS JOIN (
            SELECT DISTINCT
                YEAR(date_arrivee)  AS annee,
                MONTH(date_arrivee) AS mois
            FROM vw_reservations_propres
        ) v
    ) src
    GROUP BY hotel_id
)
SELECT
    h.nom                                                  AS hotel,
    h.etoiles,
    FORMAT(a.adr, 'N0')                                    AS adr_fcfa,
    ROUND(a.nuits_vendues * 100.0
          / NULLIF(c.cap_totale, 0), 2)                    AS taux_occupation_pct,
    FORMAT(
        ROUND(a.adr * (a.nuits_vendues * 100.0
              / NULLIF(c.cap_totale, 0)) / 100.0, 0)
    , 'N0')                                                AS revpar_fcfa,
    RANK() OVER (
        ORDER BY a.adr * (a.nuits_vendues * 100.0
                 / NULLIF(c.cap_totale, 0)) / 100.0 DESC
    )                                                      AS rang
FROM adr_global a
JOIN cap_globale c ON a.hotel_id = c.hotel_id
JOIN hotels h      ON a.hotel_id = h.hotel_id
ORDER BY rang;

-- Resultat attendu :
-- HotelChain Douala   | ADR 127 411 | Taux 6.40% | RevPAR 8 154  | Rang 1
-- HotelChain Cocody   | ADR 116 382 | Taux 5.53% | RevPAR 6 436  | Rang 2
-- HotelChain Dakar    | ADR 139 099 | Taux 4.49% | RevPAR 6 246  | Rang 3
-- HotelChain Plateau  | ADR 138 329 | Taux 3.82% | RevPAR 5 284  | Rang 4
-- HotelChain Accra    | ADR 175 203 | Taux 2.77% | RevPAR 4 853  | Rang 5


### INTERPRETATION

| Hotel | Etoiles | ADR | Taux occupation | RevPAR | Rang |
|---|---|---|---|---|---|
| HotelChain Douala | 3 ⭐ | 105 248 FCFA | 7.74% | **8 151 FCFA** | **1** |
| HotelChain Cocody | 3 ⭐ | 96 100 FCFA | 6.70% | 6 440 FCFA | 2 |
| HotelChain Dakar | 4 ⭐ | 113 606 FCFA | 5.49% | 6 239 FCFA | 3 |
| HotelChain Plateau | 4 ⭐ | 114 083 FCFA | 4.63% | 5 287 FCFA | 4 |
| HotelChain Accra | 5 ⭐ | 142 970 FCFA | 3.39% | **4 851 FCFA** | **5** |

### METIER — 4 enseignements cles

**1. Le RevPAR recompense le remplissage, pas le prix**

Douala (3 etoiles, ADR 105 248 FCFA) bat Accra (5 etoiles, ADR 142 970 FCFA) au RevPAR de **68% a 40%** — soit +3 300 FCFA par chambre disponible. Douala compense son tarif modeste par un taux d'occupation 2.3x superieur (7.74% vs 3.39%). C'est la demonstration que remplir ses chambres a prix moyen est plus rentable que de les garder vides a prix eleve.

**2. Le paradoxe des etoiles**

Les deux hotels 3 etoiles (Douala, Cocody) occupent les 2 premieres places du RevPAR, devant les 4 et 5 etoiles. Explication : un client qui choisit un 3 etoiles a des attentes calibrees et est plus facile a satisfaire. Un client 5 etoiles a des exigences tres elevees qui reduisent le taux de conversion a la reservation. Accra doit donc travailler **deux fois plus** : justifier son positionnement premium ET attirer suffisamment de clients.

**3. Dakar sous-exploite son ADR**

Dakar a un ADR de 113 606 FCFA (le 3e plus eleve) mais un taux d'occupation de seulement 5.49% — en dessous de Cocody qui facture moins cher (96 100 FCFA) mais affiche 6.70% d'occupation. Dakar a une politique tarifaire trop elevee pour son marche. Une baisse moderee de l'ADR pourrait significativement augmenter son RevPAR.

**4. Simulation Accra — combien faudrait-il gagner en occupation ?**

```
Situation actuelle   : RevPAR = 142 970 x 3.39% = 4 851 FCFA
Objectif = Douala    : RevPAR cible = 8 151 FCFA
Taux necessaire (sans toucher a l ADR) : 8 151 / 142 970 = 5.70%
Soit +2.31 points d occupation a gagner (+68% de remplissage)

Alternative : baisser l ADR de 15%
ADR nouveau = 142 970 x 0.85 = 121 525 FCFA
Taux necessaire pour egaler Douala : 8 151 / 121 525 = 6.71%
```
Le levier prix seul ne suffit pas. Accra doit combiner **baisse tarifaire moderee + actions commerciales** (partenariats corporate, packages conference, visibilite OTA) pour atteindre un taux d'occupation comparable aux hotels 3-4 etoiles du groupe.

In [ ]:
-- ADR par type de chambre (pour calibrer la politique tarifaire)
SELECT
    c.type_chambre,
    COUNT(DISTINCT c.chambre_id)            AS nb_chambres_parc,
    COUNT(r.reservation_id)                 AS nb_reservations,
    ROUND(AVG(p.montant / NULLIF(r.nb_nuits, 0)), 0) AS adr_reel,
    ROUND(AVG(c.prix_nuit), 0)              AS prix_catalogue,
    ROUND(
        (AVG(p.montant / NULLIF(r.nb_nuits, 0))
         - AVG(c.prix_nuit))
        * 100.0 / NULLIF(AVG(c.prix_nuit), 0)
    , 1)                                    AS ecart_catalogue_pct
FROM vw_reservations_propres r
JOIN chambres c   ON r.chambre_id    = c.chambre_id
JOIN vw_paiements_valides p
                  ON r.reservation_id = p.reservation_id
WHERE r.statut IN ('Terminee', 'En cours')
GROUP BY c.type_chambre
ORDER BY adr_reel DESC;


### INTERPRETATION

| Type | Nb chambres | Nb reservations | ADR reel | Prix catalogue | Ecart |
|---|---|---|---|---|---|
| Suite | 49 | 819 | 294 491 FCFA | 393 462 FCFA | **-25.2%** |
| Deluxe | 110 | 1 832 | 149 571 FCFA | 202 092 FCFA | **-26.0%** |
| Superieure | 157 | 2 666 | 100 043 FCFA | 135 063 FCFA | **-25.9%** |
| Standard | 199 | 3 633 | 66 284 FCFA | 88 525 FCFA | **-25.1%** |

### METIER — 3 enseignements

**1. Un ecart catalogue systematique de ~25%**

L'ecart entre le prix catalogue et l'ADR reel est remarquablement stable : entre -25.1% et -26.0% selon le type de chambre. Ce n'est pas du hasard — c'est la signature des commissions OTA (Booking.com -10%, Expedia -12%, Corporate -15%) cumulees aux remises directes. La coherence de cet ecart sur tous les types de chambre indique une politique tarifaire bien maitrisee : aucun segment n'est sur-remise par rapport aux autres.

**2. Les Suites sont sous-reservees**

49 Suites pour 819 reservations = **16.7 reservations par chambre** sur 30 mois, soit une reservation toutes les 7 semaines par Suite. A titre de comparaison, les Standard font 18.3 reservations par chambre. Les Suites tournent moins, ce qui est attendu vu leur prix, mais l'ecart catalogue (-25.2%) montre qu'elles ne beneficient pas de plus de remises que les autres types. Le probleme est la demande, pas la tarification.

**3. Le manque a gagner reel — cout des remises et commissions**

```sql
-- Si toutes les reservations se faisaient au prix catalogue :
Suite      : 819  x (393 462 - 294 491) =  +81 M FCFA en remises/commissions
Deluxe     : 1832 x (202 092 - 149 571) =  +96 M FCFA
Superieure : 2666 x (135 063 - 100 043) =  +93 M FCFA
Standard   : 3633 x ( 88 525 -  66 284) =  +81 M FCFA
                                  TOTAL  = ~351 M FCFA perdus en 30 mois
```
Ce montant (~351 M FCFA sur 30 mois, soit ~140 M FCFA/an) represente le cout du recours aux OTA et aux remises commerciales. Chaque reservation directe supplementaire recupere entre **22 000 et 99 000 FCFA** de marge nette selon le type de chambre. C'est le chiffre a presenter a la direction pour justifier un investissement dans le canal direct (refonte site web, programme de fidelite, CRM).

---
## Etape 6 — Segmentation clients et valeur client (CLV)

### METHODE — NTILE() et PERCENT_RANK()

**NTILE(n)** divise les lignes en n groupes de taille egale (quartiles si n=4). Le groupe 4 contient les clients les plus precieux (revenu le plus eleve).

**PERCENT_RANK()** calcule le rang relatif de chaque ligne entre 0 et 1. Un client avec PERCENT_RANK() > 0.9 est dans le top 10%.

```
NTILE(4) OVER (ORDER BY revenu DESC)
--> Quartile 4 = top 25% des clients (les plus riches en revenu)

PERCENT_RANK() OVER (ORDER BY revenu DESC)
--> 0.0 = le client qui depense le plus
    0.9 = seuil top 10%
    1.0 = le client qui depense le moins
```

In [ ]:
-- Requete 1/2 : detail par client (une ligne par client)
-- Utile pour identifier des clients specifiques, exporter vers CRM, ou filtrer un segment
-- Pour la synthese agregee par quartile, voir la requete suivante
-- Valeur client (CLV) et segmentation en quartiles
-- TRY_CAST(csat AS INT) car csat est VARCHAR dans la table
WITH clv AS (
    SELECT
        c.client_id,
        c.prenom + ' ' + c.nom           AS client_nom,
        c.nationalite,
        c.client_fidele,
        COUNT(r.reservation_id)          AS nb_sejours,
        ROUND(SUM(p.montant), 0)         AS revenu_total,
        ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 1) AS duree_moy,
        ROUND(AVG(TRY_CAST(r.csat AS FLOAT)), 1) AS csat_moy
    FROM vw_clients_propres c
    JOIN vw_reservations_propres r
         ON c.client_id = r.client_id
    LEFT JOIN vw_paiements_valides p
         ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY c.client_id, c.prenom, c.nom, c.nationalite, c.client_fidele
)
SELECT
    client_id,
    client_nom,
    nationalite,
    client_fidele,
    nb_sejours,
    revenu_total,
    duree_moy,
    csat_moy,
    NTILE(4) OVER (ORDER BY revenu_total DESC)         AS quartile,
    CASE
        WHEN PERCENT_RANK() OVER (ORDER BY revenu_total DESC) <= 0.10
        THEN 'VIP Top 10%'
        WHEN PERCENT_RANK() OVER (ORDER BY revenu_total DESC) <= 0.25
        THEN 'Premium'
        WHEN PERCENT_RANK() OVER (ORDER BY revenu_total DESC) <= 0.75
        THEN 'Standard'
        ELSE 'Occasionnel'
    END                                                AS segment
FROM clv
ORDER BY revenu_total DESC;


In [ ]:
-- Requete 2/2 : synthese par quartile
-- C'est CETTE requete qui correspond au tableau de l'interpretation ci-dessous
WITH clv AS (
    SELECT
        c.client_id,
        c.client_fidele,
        COUNT(r.reservation_id)                  AS nb_sejours,
        ROUND(SUM(p.montant), 0)                 AS revenu_total,
        ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 1) AS duree_moy,
        ROUND(AVG(TRY_CAST(r.csat AS FLOAT)), 1) AS csat_moy
    FROM vw_clients_propres c
    JOIN vw_reservations_propres r  ON c.client_id    = r.client_id
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY c.client_id, c.client_fidele
),
clv_avec_quartile AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY revenu_total DESC) AS quartile
    FROM clv
),
revenu_global AS (
    SELECT SUM(revenu_total) AS total FROM clv
)
SELECT
    CASE quartile
        WHEN 1 THEN 'Q4 (top 25%)'
        WHEN 2 THEN 'Q3'
        WHEN 3 THEN 'Q2'
        WHEN 4 THEN 'Q1 (bas 25%)'
    END                                                    AS quartile_label,
    COUNT(*)                                               AS nb_clients,
    FORMAT(ROUND(AVG(revenu_total), 0), 'N0')              AS revenu_moyen_fcfa,
    ROUND(AVG(nb_sejours), 0)                              AS sejours_moyens,
    ROUND(AVG(duree_moy), 1)                               AS duree_moy_nuits,
    ROUND(AVG(csat_moy), 2)                                AS csat_moyen,
    ROUND(SUM(revenu_total) * 100.0
          / (SELECT total FROM revenu_global), 1)          AS pct_revenu_total
FROM clv_avec_quartile
GROUP BY quartile
ORDER BY quartile ASC;


### INTERPRETATION

| Quartile | Nb clients | Revenu moyen | Sejours moyens | % revenu total |
|---|---|---|---|---|
| Q4 (top 25%) | 683 | **2 051 193 FCFA** | 4 sejours | ~50% |
| Q3 | 682 | 1 076 593 FCFA | 3 sejours | ~26% |
| Q2 | 681 | 656 252 FCFA | 2 sejours | ~16% |
| Q1 (bas 25%) | 681 | 279 573 FCFA | 1 sejour | ~7% |

### METIER — La loi de Pareto confirmee

Le top 25% des clients (Q4, 683 clients) genere **~50% du revenu total**. C'est la loi de Pareto (80/20) adaptee au secteur hotelier. Ces 683 clients VIP meritent un programme de fidelisation dedie : upgrade automatique, late checkout, taux preferentiel. Le cout d'un geste commercial est negligeable face a leur valeur vie.

**Clients fideles vs nouveaux :**
- Reservations fideles : **23.9%** du total
- CSAT fideles (4.00) vs nouveaux (4.02) — quasi-identique
- Duree sejour identique (2.71 nuits)

Resultat contre-intuitif : les clients fideles n'ont pas un CSAT superieur. Ils connaissent l'hotel et ont des attentes calibrees — ni surpris en bien, ni deçus. Le programme de fidelisation doit travailler sur l'experimentation pour creer de vraies surprises positives.

In [ ]:
-- Analyse des clients fidelesles vs nouveaux
SELECT
    CASE WHEN c.client_fidele = 1 THEN 'Fidele' ELSE 'Nouveau' END AS segment,
    COUNT(DISTINCT r.reservation_id)                               AS nb_reservations,
    ROUND(COUNT(DISTINCT r.reservation_id) * 100.0
          / SUM(COUNT(DISTINCT r.reservation_id)) OVER(), 1)       AS part_pct,
    ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 2)                      AS duree_moy,
    ROUND(AVG(TRY_CAST(r.csat AS FLOAT)), 2)                      AS csat_moy,
    ROUND(AVG(p.montant), 0)                                       AS revenu_moy_res
FROM vw_clients_propres c
JOIN vw_reservations_propres r  ON c.client_id    = r.client_id
LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
WHERE r.statut IN ('Terminee', 'En cours')
GROUP BY c.client_fidele;


---
## Etape 7 — Heatmap saisonnalite (PIVOT) et procedure stockee

### METHODE — L'operateur PIVOT

`PIVOT` transforme les valeurs d'une colonne en colonnes separees. C'est l'equivalent du `pivot()` de pandas ou du tableau croise dynamique Excel.

**Structure obligatoire :**
```sql
SELECT * FROM (
    -- 1. Sous-requete : colonnes ID + dimension + valeur
    SELECT hotel, mois, nb_nuits FROM ...
) src
PIVOT (
    SUM(nb_nuits)           -- Agregation
    FOR mois                -- Colonne a pivoter
    IN ([1],[2],[3],...,[12]) -- Valeurs a transformer en colonnes
) pvt;
```

**Limitation SQL Server :** les valeurs du `IN` doivent etre connues a l'ecriture de la requete (pas de pivot dynamique natif). Pour un pivot dynamique (colonnes inconnues a l'avance), il faut utiliser du SQL dynamique (`EXEC sp_executesql`).

In [ ]:
-- PIVOT : taux d'occupation par hotel x mois (annee 2023)
SELECT *
FROM (
    SELECT
        h.nom                      AS hotel,
        MONTH(r.date_arrivee)      AS mois,
        -- Calcul taux occupation pour le PIVOT (nuits vendues / capacite)
        ROUND(r.nb_nuits * 100.0
              / (h.nb_chambres
                 * DAY(EOMONTH(r.date_arrivee))), 2) AS taux_unit
    FROM vw_reservations_propres r
    JOIN hotels h ON r.hotel_id = h.hotel_id
    WHERE r.statut IN ('Terminee', 'En cours')
      AND YEAR(r.date_arrivee) = 2023
) src
PIVOT (
    SUM(taux_unit)
    FOR mois IN (
        [1],[2],[3],[4],[5],[6],
        [7],[8],[9],[10],[11],[12]
    )
) pvt
ORDER BY hotel;


### INTERPRETATION — Heatmap saisonnalite 2023

Le pivot revele les patterns saisonniers par hotel sur 2023 :

| Hotel | Meilleur mois | Pire mois | Amplitude |
|---|---|---|---|
| Douala | Aout (8.5%) | Septembre (5.5%) | 3.0 pts |
| Cocody | Aout (7.3%) | Juin (3.6%) | 3.7 pts |
| Dakar | Juillet (6.9%) | Fevrier (3.7%) | 3.2 pts |
| Plateau | Janvier (6.4%) | Septembre (3.3%) | 3.1 pts |
| Accra | Aout (3.7%) | Mars (2.1%) | 1.6 pts |

**Juillet-Aout** est la haute saison pour tous les hotels (pic estival). **Fevrier-Mars** est systematiquement en basse saison. Cette heatmap sera reproduite directement dans Power BI (page 2 du dashboard) avec mise en forme conditionnelle rouge/orange/vert.

In [ ]:
-- PROCEDURE STOCKEE : rapport complet d'un hotel
CREATE OR ALTER PROCEDURE sp_rapport_hotel
    @hotel_id  VARCHAR(10),
    @annee     INT = 2023
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @nom_hotel VARCHAR(100);
    SELECT @nom_hotel = nom FROM hotels WHERE hotel_id = @hotel_id;

    PRINT '======================================';
    PRINT 'RAPPORT : ' + @nom_hotel;
    PRINT 'ANNEE   : ' + CAST(@annee AS VARCHAR);
    PRINT '======================================';

    -- KPIs annuels de l hotel
    -- TRY_CAST(r.csat AS FLOAT) car csat est VARCHAR(5) dans la table
    SELECT
        @nom_hotel                                  AS hotel,
        @annee                                      AS annee,
        COUNT(r.reservation_id)                     AS nb_reservations,
        ROUND(SUM(p.montant), 0)                    AS revenu_total,
        ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 2)    AS duree_moy,
        ROUND(AVG(TRY_CAST(r.csat AS FLOAT)), 2)    AS csat_moyen,
        ROUND(SUM(r.nb_nuits) * 100.0
              / (h.nb_chambres * 365), 1)           AS taux_occupation_annuel,
        ROUND(SUM(p.montant)
              / NULLIF(SUM(r.nb_nuits), 0), 0)      AS adr,
        ROUND((SUM(r.nb_nuits) * 100.0
               / (h.nb_chambres * 365))
              * SUM(p.montant)
              / NULLIF(SUM(r.nb_nuits), 0)
              / 100.0, 0)                           AS revpar
    FROM vw_reservations_propres r
    JOIN hotels h          ON r.hotel_id       = h.hotel_id
    LEFT JOIN vw_paiements_valides p
                           ON r.reservation_id = p.reservation_id
    WHERE r.hotel_id = @hotel_id
      AND YEAR(r.date_arrivee) = @annee
      AND r.statut IN ('Terminee', 'En cours')
    GROUP BY h.nb_chambres;

    -- Top 3 canaux pour cet hotel
    SELECT TOP 3
        r.canal,
        COUNT(*)                AS nb_reservations,
        ROUND(SUM(p.montant),0) AS revenu
    FROM vw_reservations_propres r
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.hotel_id = @hotel_id
      AND YEAR(r.date_arrivee) = @annee
      AND r.statut IN ('Terminee', 'En cours')
    GROUP BY r.canal
    ORDER BY revenu DESC;
END;
GO

-- Test pour HotelChain Plateau (HTL001) en 2023
EXEC sp_rapport_hotel @hotel_id = 'HTL001', @annee = 2023;

-- Test pour HotelChain Accra (HTL005) en 2023
EXEC sp_rapport_hotel @hotel_id = 'HTL005', @annee = 2023;


### INTERPRETATION — Rapport HTL001 Plateau 2023

| KPI | Valeur |
|---|---|
| Reservations | **766** |
| CSAT moyen | 3.99 / 5 |
| Duree moyenne | 2.84 nuits |
| Taux occupation annuel | **4.0%** |

### METHODE — Pourquoi les procedures stockees ?

Une procedure stockee offre 4 avantages :
1. **Reutilisabilite** : un seul `EXEC sp_rapport_hotel @hotel_id='HTL001'` au lieu de reecrire 30 lignes SQL a chaque fois
2. **Securite** : on peut donner acces a la procedure sans donner acces aux tables
3. **Performance** : le plan d'execution est compile et mis en cache
4. **Standardisation** : tous les directeurs d'hotel recoivent le meme rapport, produit par la meme logique

---
## Etape 8 — Analyse des services et revenus extras

### METHODE

Les revenus extras (restaurant, spa, minibar...) sont un levier important car ils n'impliquent pas de nouvelle reservation. On calcule le ratio extras/chambre pour identifier les opportunites d'upselling.

In [ ]:
-- Revenus extras par categorie et par hotel
SELECT
    h.nom                                        AS hotel,
    s.categorie,
    COUNT(*)                                     AS nb_consommations,
    ROUND(SUM(s.montant), 0)                     AS revenu_extras,
    ROUND(AVG(s.montant), 0)                     AS ticket_moyen,
    RANK() OVER (
        PARTITION BY h.nom
        ORDER BY SUM(s.montant) DESC
    )                                            AS rang_dans_hotel
FROM services s
JOIN hotels h ON s.hotel_id = h.hotel_id
GROUP BY h.nom, s.categorie
ORDER BY h.nom, revenu_extras DESC;


In [ ]:
-- Ratio extras / revenu chambre par hotel
SELECT
    h.nom                                        AS hotel,
    ROUND(SUM(p.montant), 0)                     AS revenu_chambre,
    ROUND(SUM(s.montant), 0)                     AS revenu_extras,
    ROUND(SUM(s.montant) * 100.0
          / NULLIF(SUM(p.montant), 0), 1)        AS ratio_extras_pct
FROM hotels h
LEFT JOIN vw_reservations_propres r  ON h.hotel_id = r.hotel_id
LEFT JOIN vw_paiements_valides p     ON r.reservation_id = p.reservation_id
LEFT JOIN services s                 ON r.reservation_id = s.reservation_id
WHERE r.statut IN ('Terminee', 'En cours')
GROUP BY h.nom
ORDER BY ratio_extras_pct DESC;


-- Revenus extras par categorie
SELECT
    s.categorie,
    COUNT(*)                                     AS nb_consommations,
    ROUND(SUM(s.montant), 0)                     AS revenu_extras,
    ROUND(AVG(s.montant), 0)                     AS ticket_moyen
FROM services s
JOIN hotels h ON s.hotel_id = h.hotel_id
GROUP BY  s.categorie
ORDER BY  revenu_extras DESC;


### INTERPRETATION

| Categorie | Revenu total | Ticket moyen | Nb transactions |
|---|---|---|---|
| Restaurant | **68.9 M FCFA** | 24 707 FCFA | 2 790 |
| Salle conference | **65.1 M FCFA** | **132 504 FCFA** | 491 |
| Spa & Bien-etre | 52.6 M FCFA | 74 839 FCFA | 703 |
| Room service | 33.5 M FCFA | 19 552 FCFA | 1 714 |
| Transport | 29.1 M FCFA | 32 669 FCFA | 890 |

**Extras total : 278 M FCFA = 9.7% du revenu total chambres.** C'est un ratio faible — la norme pour un 4-5 etoiles est 15-25%. La Salle conference a le ticket moyen le plus eleve (132 504 FCFA) mais seulement 491 consommations. C'est le service a developper en priorite car chaque conference vendue genere 7x plus qu'un repas au restaurant.

---
## Bilan du Notebook 2

### Objets SQL crees dans HotelChainDB

| Objet | Type | Role |
|---|---|---|
| `vw_clients_propres` | Vue | Exclut doublons et ages aberrants |
| `vw_reservations_propres` | Vue | Exclut montants negatifs et nuits=0 |
| `vw_paiements_valides` | Vue | Garde uniquement statut Valide et montant>0 |
| `sp_rapport_hotel` | Procedure stockee | Rapport KPIs par hotel et par annee |

### Reponses aux 5 questions business

| Question | Reponse |
|---|---|
| Hotel meilleur taux occupation | **HotelChain Douala (6.4% moyen, 8.6% max)** |
| Meilleur mois toutes saisons | **Juillet-Aout** (haute saison) |
| Canal le plus rentable | **Direct (913 M FCFA, 0% commission)** |
| Meilleur RevPAR | **Douala (8 156 FCFA/chambre)** |
| Top 25% clients = ? du revenu | **~50% du revenu total** (loi de Pareto) |

### Notions SQL maitrisees

- `CREATE OR ALTER VIEW` — vues de nettoyage non destructives
- `CASE WHEN ... END` — agregation conditionnelle
- `RANK() OVER (PARTITION BY ... ORDER BY ...)` — classement dans un groupe
- `LAG() OVER (PARTITION BY ... ORDER BY ...)` — valeur du mois precedent
- `NTILE(4)` et `PERCENT_RANK()` — segmentation statistique
- `PIVOT ... FOR ... IN (...)` — tableau croise dynamique
- `CREATE OR ALTER PROCEDURE` — procedure stockee parametree
- `NULLIF(expr, 0)` — protection contre la division par zero
- `DAY(EOMONTH(date))` — nombre de jours dans un mois

**Pour le Notebook 3 :** Connecter Power BI a SQL Server en Import Mode, utiliser les vues comme sources, creer les 12 mesures DAX et construire le dashboard 5 pages.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.